# 03 — TF-IDF Feature Engineering

**Research**: Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF

**Objective**: Transform preprocessed Indonesian text into TF-IDF numerical feature vectors for Naive Bayes, Logistic Regression, and SVM.

**Pipeline**: Load → Split (80/20 stratified) → Fit TF-IDF on train → Transform → Save artifacts

---

## 1. Setup

In [1]:
import sys
import os

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import project modules
from src.config.settings import (
    CLEAN_DATASET_PATH,
    FEATURE_REPORTS_DIR,
)
from src.config.tfidf_config import TfidfConfig
from src.data.load_dataset import load_dataset
from src.feature_engineering.tfidf_pipeline import (
    run_tfidf_pipeline,
    generate_feature_report,
    save_feature_report,
)
from src.feature_engineering.feature_statistics import (
    get_top_tfidf_terms,
    export_feature_statistics,
    export_feature_importance_preview,
)
from src.feature_engineering.feature_validator import validate_features
from src.feature_engineering.persistence import save_tfidf_artifacts

print('Setup complete.')

Setup complete.


---
## 2. Load Preprocessed Dataset

In [2]:
df = load_dataset(filepath=CLEAN_DATASET_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

✓ Dataset loaded successfully: 41,556 rows, 3 columns
Shape: (41556, 3)
Columns: ['text', 'label', 'text_clean']


,text,label,text_clean
0,setiap orang adalah seorang gadis yang akan me...,harassment,orang gadis ganggu sekolah
1,bahwa pos ab kpop stans pergi ke sekolah bersa...,harassment,pos ab kpop stans pergi sekolah bersamasama or...
2,karena beberapa orang tidak ada yang lebih bai...,harassment,orang ganggu sekolah outgrew
3,bro aku pelatih jv tahun lalu di skyline dan a...,harassment,bro latih jv skyline ubah air anggur menang ta...
4,wanitawanita ini benarbenar mengingatkan saya ...,harassment,wanitawanita benarbenar anak ayam sma tindas d...


---
## 3. Validate Dataset

In [3]:
# Quick validation
assert 'text_clean' in df.columns, 'Missing text_clean column. Run preprocessing first.'
assert 'label' in df.columns, 'Missing label column.'
print(f'✓ Dataset validated: {len(df):,} rows')
print(f'  text_clean nulls: {df["text_clean"].isnull().sum()}')
print(f'  label nulls: {df["label"].isnull().sum()}')
print(f'  Labels: {sorted(df["label"].unique())}')

✓ Dataset validated: 41,556 rows
  text_clean nulls: 32
  label nulls: 0
  Labels: ['harassment', 'hate_speech', 'insult', 'normal', 'sexually_explicit', 'threat']


---
## 4. Configure TF-IDF

In [4]:
config = TfidfConfig(
    max_features=None,       # No limit
    min_df=2,                # Ignore terms in < 2 documents
    max_df=0.95,             # Ignore terms in > 95% documents
    ngram_range=(1, 1),      # Unigrams
    norm='l2',
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=True,       # Use log(1+tf)
    analyzer='word',
    lowercase=False,         # Already lowercased by preprocessing
    test_size=0.20,          # 80/20 split per TRD
    random_state=42,         # Per TRD
    stratify=True,           # Maintain class distribution
)

print('TF-IDF Configuration:')
for k, v in config.to_dict().items():
    print(f'  {k}: {v}')

TF-IDF Configuration:
  max_features: None
  min_df: 2
  max_df: 0.95
  ngram_range: (1, 1)
  norm: l2
  use_idf: True
  smooth_idf: True
  sublinear_tf: True
  analyzer: word
  lowercase: False
  token_pattern: (?u)\b\w+\b
  test_size: 0.2
  random_state: 42
  stratify: True


---
## 5. Run TF-IDF Pipeline

This will:
1. Split dataset (80/20 stratified)
2. Fit TF-IDF on training data **only**
3. Transform both train and test sets
4. Compute statistics

In [5]:
X_train, X_test, y_train, y_test, vectorizer, stats = run_tfidf_pipeline(
    df=df,
    config=config,
    text_col='text_clean',
    label_col='label',
)

  TF-IDF Feature Engineering Pipeline

[1/4] Splitting dataset...
✓ Dataset split: Train=33,244, Test=8,312
  Split ratio: 80.0% / 20.0%

[2/4] Creating TF-IDF vectorizer...
  Config: max_features=None, min_df=2, max_df=0.95, ngram_range=(1, 1)

[3/4] Fitting and transforming...
  ✓ Fitted on X_train: (33244, 22695)
  ✓ Transformed X_test: (8312, 22695)
  Vocabulary size: 22,695

[4/4] Computing statistics...
  Matrix density: 0.000661
  Matrix sparsity: 0.999339
  Avg features/doc: 15.0

  ✓ TF-IDF Pipeline Complete


---
## 6. Feature Statistics

In [6]:
print('Feature Statistics:')
print(f'  Vocabulary Size:          {stats["vocabulary_size"]:,}')
print(f'  Number of Features:       {stats["n_features"]:,}')
print(f'  Training Matrix Shape:    {stats["train_shape"]}')
print(f'  Testing Matrix Shape:     {stats["test_shape"]}')
print(f'  Matrix Density:           {stats["matrix_density"]:.6f}')
print(f'  Matrix Sparsity:          {stats["matrix_sparsity"]:.6f}')
print(f'  Avg Features/Document:    {stats["avg_features_per_document"]:.2f}')
print(f'  Min Features/Document:    {stats["min_features_per_document"]}')
print(f'  Max Features/Document:    {stats["max_features_per_document"]}')

Feature Statistics:
  Vocabulary Size:          22,695
  Number of Features:       22,695
  Training Matrix Shape:    (33244, 22695)
  Testing Matrix Shape:     (8312, 22695)
  Matrix Density:           0.000661
  Matrix Sparsity:          0.999339
  Avg Features/Document:    14.99
  Min Features/Document:    0
  Max Features/Document:    550


In [7]:
# Top TF-IDF terms
top_terms = get_top_tfidf_terms(vectorizer, X_train, n=30)

print('Top 20 TF-IDF Terms (by avg score):')
for i, (term, score) in enumerate(top_terms[:20], 1):
    print(f'  {i:>2}. {term:>20}: {score:.6f}')

Top 20 TF-IDF Terms (by avg score):
   1.               israel: 0.024755
   2.                 gila: 0.021286
   3.                china: 0.014537
   4.                orang: 0.014362
   5.                    x: 0.012996
   6.            indonesia: 0.012417
   7.            palestina: 0.011617
   8.                 gaza: 0.010399
   9.               serang: 0.007899
  10.               dukung: 0.007302
  11.             presiden: 0.007209
  12.               jokowi: 0.007145
  13.                hamas: 0.007124
  14.               negara: 0.007015
  15.                    f: 0.006869
  16.                 anak: 0.006869
  17.                   xf: 0.006521
  18.                    n: 0.006384
  19.                warga: 0.006383
  20.                islam: 0.006363


---
## 7. Validate Features

In [8]:
validation = validate_features(X_train, X_test, y_train, y_test, vectorizer)
print(validation['summary'])

Feature Validation Report
--------------------------------------------------
  ✓ vocabulary_not_empty
  ✓ X_train_not_empty
  ✓ X_test_not_empty
  ✓ feature_dims_match
  ✓ train_labels_match
  ✓ test_labels_match
  ✓ label_sets_consistent
  ✓ no_nan_values
  ✓ no_inf_values
  ✓ no_missing_labels
--------------------------------------------------
  Status: PASS


---
## 8. Save Artifacts

In [9]:
print('Saving artifacts...')
saved_paths = save_tfidf_artifacts(
    vectorizer=vectorizer,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    overwrite=True,
)
print(f'\n✓ All artifacts saved.')

Saving artifacts...
  ✓ Vectorizer saved: /home/zapp/Kampus/PM/models/tfidf_vectorizer.joblib
  ✓ X_train saved: /home/zapp/Kampus/PM/dataset/processed/X_train.npz (shape=(33244, 22695))
  ✓ X_test saved: /home/zapp/Kampus/PM/dataset/processed/X_test.npz (shape=(8312, 22695))
  ✓ y_train saved: /home/zapp/Kampus/PM/dataset/processed/y_train.csv (33,244 rows)
  ✓ y_test saved: /home/zapp/Kampus/PM/dataset/processed/y_test.csv (8,312 rows)

✓ All artifacts saved.


---
## 9. Generate Reports

In [10]:
print('Generating reports...\n')

# Feature summary report (markdown)
report_content = generate_feature_report(
    stats=stats,
    config=config,
    validation=validation,
    top_terms=top_terms,
)
save_feature_report(
    report_content,
    os.path.join(FEATURE_REPORTS_DIR, 'feature_summary.md'),
)

# Feature statistics CSV
export_feature_statistics(
    stats,
    os.path.join(FEATURE_REPORTS_DIR, 'feature_statistics.csv'),
)

# Feature importance preview CSV
export_feature_importance_preview(
    top_terms,
    os.path.join(FEATURE_REPORTS_DIR, 'feature_importance_preview.csv'),
)

print(f'\n✓ All reports generated.')

Generating reports...

  ✓ Report saved: /home/zapp/Kampus/PM/reports/feature_engineering/feature_summary.md
  ✓ Statistics exported: /home/zapp/Kampus/PM/reports/feature_engineering/feature_statistics.csv
  ✓ Feature importance preview exported: /home/zapp/Kampus/PM/reports/feature_engineering/feature_importance_preview.csv

✓ All reports generated.


---
## Summary

Feature engineering is complete. Generated outputs:

**Artifacts**:
- `models/tfidf_vectorizer.joblib` — Fitted TF-IDF vectorizer
- `dataset/processed/X_train.npz` — Training features
- `dataset/processed/X_test.npz` — Testing features
- `dataset/processed/y_train.csv` — Training labels
- `dataset/processed/y_test.csv` — Testing labels

**Reports**:
- `reports/feature_engineering/feature_summary.md`
- `reports/feature_engineering/feature_statistics.csv`
- `reports/feature_engineering/feature_importance_preview.csv`

**Next Step**: `04_model_training.ipynb` — Train NB, LR, SVM